In [158]:
import pandas as pd
import numpy as np

def data_harmonization(dc):
    
    aa_dict = {'A': 'Ala','R': 'Arg','N': 'Asn','D': 'Asp','C': 'Cys','E': 'Glu','Q': 'Gln','G': 'Gly','H': 'His',
           'I': 'Ile','L': 'Leu','K': 'Lys','M': 'Met','F': 'Phe','P': 'Pro','S': 'Ser','T': 'Thr','W': 'Trp',
           'Y': 'Tyr','V': 'Val', '=': '=','*': 'Ter', "~": 'Del', '-':'Del','STOP': '*'}

    reversed_aa_dict = {v: k for k, v in aa_dict.items()}
    
    x = dc['x']
    
    y = dc['y']
    
    df1 = pd.read_excel(x, header = y)
    
    #create an empty dataframe with column names as follows

    pillar_data = pd.DataFrame(columns=["Dataset", "Gene", "HGNC_id","Chrom","hg19_pos","hg38_start","ref_allele",
                                    "alt_allele","auth_transcript_id","transcript_pos","transcript_ref","transcript_alt",
                                   "aa_pos","aa_ref","aa_alt","hgvs_c","hgvs_p","consequence","auth_reported_score",
                                   "auth_reported_rep_score","auth_reported_func_class","auth_reported_normal_min",
                                   "auth_reported_normal_max","auth_reported_abnormal_min","auth_reported_abnormal_max",
                                   "splice_measure","gnomad_MAF","clinvar_sig","clinvar_star",
                                    "clinvar_date_last_reviewed","nucleotide_or_aa"])
    
    num_rows = len(df1)
    pillar_data = pd.DataFrame(index=range(num_rows), columns=pillar_data.columns)

    for key, value in dc.items():
        if key in pillar_data.columns:
            if pd.api.types.is_scalar(value):
                pillar_data[key] = value
            else:
                pillar_data[key] = value.values
                    
                    
    #converting syntax such as p.M1L or M1L to p.Met1Leu
    if dc['hgvs_p_conversion'] == 'Yes':
        hgvs_pro_list = list()
        for index, row in pillar_data.iterrows():
            try:
                if pd.isna(row['hgvs_p']):
                    hgvs_pro = np.nan
                    hgvs_pro_list.append(hgvs_pro)
                elif row['hgvs_p'].startswith('p.'):
                    if row['hgvs_p'].endswith("="):
                        alt_a = row["hgvs_p"][2]
                        ref_a = row['hgvs_p'][2]
                        position = row['hgvs_p'][3:-1]
                    else:
                        ref_a = row['hgvs_p'][2]
                        alt_a = row['hgvs_p'][-1]
                        position = row['hgvs_p'][3:-1]
                    hgvs_pro = f"p.{aa_dict[ref_a]}{position}{aa_dict[alt_a]}"
                    hgvs_pro_list.append(hgvs_pro)
                else:
                    if row['hgvs_p'].endswith("="):
                        alt_a = row["hgvs_p"][2]
                        ref_a = row['hgvs_p'][0]
                        position = row['hgvs_p'][1:-1]
                    else:
                        ref_a = row['hgvs_p'][0]
                        alt_a = row['hgvs_p'][-1]
                        position = row['hgvs_p'][1:-1]
                    hgvs_pro = f"p.{aa_dict[ref_a]}{position}{aa_dict[alt_a]}"
                    hgvs_pro_list.append(hgvs_pro)
                    
            except KeyError as e:
                print(f"KeyError for row {index} with value {row['hgvs_p']}: {e}, {x}")
                hgvs_pro_list.append(row['hgvs_p'])
                
                continue

        pillar_data['hgvs_p'] = hgvs_pro_list
        
        

    #uses amino acid pos, ref, and alt to construct an hgvs p. format. 
    if dc['hgvs_from_aa'] == 'Yes':
        hgvs_pro_list = list()
        for index, row in pillar_data.iterrows():
            if pd.isna(row['aa_ref']) & pd.isna(row['aa_alt']):
                hgvs_pro = np.nan
                hgvs_pro_list.append(hgvs_pro)
            else:
                hgvs_pro_list.append(f"p.{aa_dict[row['aa_ref']]}{row['aa_pos']}{aa_dict[row['aa_alt']]}")
                
        pillar_data['hgvs_p'] = hgvs_pro_list

    #uses hgvs p. format to fill in amino acid pos, alt and ref columns
    if dc['aa_from_hgvs'] == 'Yes':
        aa_ref = list()
        aa_alt = list()
        aa_pos = list()
        for index, row in pillar_data.iterrows():
            try:
                if pd.isna(row['hgvs_p']):
                    aa_ref.append(np.nan) 
                    aa_alt.append(np.nan)
                    aa_pos.append(np.nan)
                else:
                    aa_ref.append(reversed_aa_dict[row['hgvs_p'][2:5]])
                    if row['hgvs_p'].endswith("="):
                        aa_alt.append('=')
                        aa_pos.append(row['hgvs_p'][5:-1])
                    elif row['hgvs_p'].endswith("*"):
                        aa_alt.append('*')
                        aa_pos.append(row['hgvs_p'][5:-1])
                    elif row['hgvs_p'].endswith("Ter"):
                        aa_alt.append('*')
                        aa_pos.append(row['hgvs_p'][5:-3])
                    else:
                        aa_alt.append(reversed_aa_dict[row['hgvs_p'][-3:]])
                        aa_pos.append(row['hgvs_p'][5:-3])
                        
            except KeyError as e:
                print(f"KeyError for row {index} with value {row['hgvs_p']}: {e}, {x}")
                aa_ref.append(row['hgvs_p'])
                aa_alt.append(row['hgvs_p'])
                aa_pos.append(np.nan)
                
                continue
                
        pillar_data['aa_ref'] = aa_ref
        pillar_data['aa_alt'] = aa_alt
        pillar_data['aa_pos'] = aa_pos
    
    #if hgvs_c is provided, it uses that information to populate transcript information
    if dc['transcript_from_hgvs_c'] == 'Yes':
        transcript_pos = list()
        transcript_ref = list()
        transcript_alt = list()
        for index, row in pillar_data.iterrows():
            if pd.isna(row['hgvs_c']):
                transcript_ref.append(np.nan) 
                transcript_alt.append(np.nan)
                transcript_pos.append(np.nan)
            else:
                transcript_ref.append(row['hgvs_c'][-3])
                transcript_alt.append(row['hgvs_c'][-1])
                transcript_pos.append(row['hgvs_c'][2:-3])
        pillar_data['transcript_pos'] = transcript_pos
        pillar_data['transcript_ref'] = transcript_ref
        pillar_data['transcript_alt'] = transcript_alt
                               
                
    return(pillar_data)

In [159]:
def data_harmonization_loop(dc_list):
    
    combined_dataframes = []
    
    
    for dc in dc_list:
            harmonized_df = data_harmonization(dc)
        
        
            if harmonized_df is not None and not harmonized_df.empty:
                combined_dataframes.append(harmonized_df) 
    
    
    combined_df = pd.concat(combined_dataframes, ignore_index=True) if combined_dataframes else pd.DataFrame()
    
    return combined_df

In [162]:
df1 = pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA1_Findlay_2018.xlsx", header = 2)
dc_1 = {"x": "~/Downloads/Pillar_project_data_files/BRCA1_Findlay_2018.xlsx", 
               "y": 2,
               "Dataset" :'BRCA1_Findlay_2018', 
               "Gene" : df1['gene'], 
               "HGNC_id" : 1100, 
               "Chrom" : df1['chromosome'], 
               "hg19_pos" : df1['position (hg19)'], 
               "hg38_start" : np.nan, 
               "ref_allele": df1['reference'], 
               "alt_allele": df1['alt'], 
               "auth_transcript_id" : df1['transcript_ID'], 
               "transcript_pos" : df1['transcript_position'],
               "transcript_ref" : df1['transcript_ref'], 
               "transcript_alt" : df1['transcript_alt'],
               "aa_pos": df1['aa_pos'],
               "aa_ref": df1['aa_ref'],
               "aa_alt": df1['aa_alt'],
               "hgvs_c": df1['transcript_variant'],
               "hgvs_p" : df1['protein_variant'],
               "consequence": df1['consequence'],
               "auth_reported_score": df1['function.score.mean'],
               "auth_reported_rep_score": df1[['function.score.r1', 'function.score.r2']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": df1['func.class'],
               "auth_reported_normal_min":-0.748,
               "auth_reported_normal_max":1.307,
               "auth_reported_abnormal_min":-5.651,
               "auth_reported_abnormal_max":-1.328,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#one amino acid change from reference sequence; amino acid 1613 in reference changed from S-->G in experiment
df2 = pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA1_Adamovich_2022_HDR.xlsx", header = 0)
dc_2 = {"x": "~/Downloads/Pillar_project_data_files/BRCA1_Adamovich_2022_HDR.xlsx", 
               "y": 0,
               "Dataset" :'BRCA1_Adamovich_2022_HDR', 
               "Gene" : "BRCA1", 
               "HGNC_id" : 1100, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df2['variantID'],
               "consequence": np.nan,
               "auth_reported_score": df2['FS_average_BRCA1_siRNA'],
               "auth_reported_rep_score": df2[['FS1_BRCA1_siRNA', 'FS2_BRCA1_siRNA','FS3_BRCA1_siRNA','FS4_BRCA1_siRNA']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min":-0.233,
               "auth_reported_normal_max":1.357,
               "auth_reported_abnormal_min":-3.198,
               "auth_reported_abnormal_max":-0.562,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#one amino acid change from reference sequence; amino acid 1613 in reference changed from S-->G in experiment
df3 = pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA1_Adamovich_2022_Cisplatin.xlsx", header = 0)
dc_3 = {"x": "~/Downloads/Pillar_project_data_files/BRCA1_Adamovich_2022_Cisplatin.xlsx", 
               "y": 0,
               "Dataset" :'BRCA1_Adamovich_2022_Cisplatin', 
               "Gene" : "BRCA1", 
               "HGNC_id" : 1100, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df3['variantID'],
               "consequence": np.nan,
               "auth_reported_score": df3['FS_average_BRCA1_siRNA'],
               "auth_reported_rep_score": df3[['FS1_BRCA1_siRNA', 'FS2_BRCA1_siRNA','FS3_BRCA1_siRNA','FS4_BRCA1_siRNA']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min":-0.503,
               "auth_reported_normal_max":0.942,
               "auth_reported_abnormal_min":-3.198,
               "auth_reported_abnormal_max":-0.729,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#p.Val2687A changed to p.Val2687Ala. Mistake in file from authors
df4 = pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA2_Hu_2024.xlsx", header = 2)
dc_4 = {"x": "~/Downloads/Pillar_project_data_files/BRCA2_Hu_2024.xlsx", 
               "y": 2,
               "Dataset" :'BRCA2_Hu_2024', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.3", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df4['coding nucleotide change'],
               "hgvs_p" : df4['Protein change'],
               "consequence": np.nan,
               "auth_reported_score": df4['HDR score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df4['HDR function'],
               "auth_reported_normal_min":2.5,
               "auth_reported_normal_max":np.nan,
               "auth_reported_abnormal_min":np.nan,
               "auth_reported_abnormal_max":1.49,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

df5 = pd.read_excel("~/Downloads/Pillar_project_data_files/MSH2_Jia_2021.xlsx", header = 0)
dc_5 = {"x": "~/Downloads/Pillar_project_data_files/MSH2_Jia_2021.xlsx", 
               "y": 0,
               "Dataset" :'MSH2_Jia_2021', 
               "Gene" : "MSH2", 
               "HGNC_id" : 7325, 
               "Chrom" : 2, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000251.2", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df5["Position"],
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df5['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df5['LOF score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": 0,
               "auth_reported_abnormal_min": 0,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df6 = pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_6 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_WAF1nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df6['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df6['WAF1nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df7 = pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_7 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_MDM2nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df7['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df7['MDM2nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}
df8 = pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_8 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_BAXnWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df8['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df8['BAXnWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}
df9= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_9 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_h1433snWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df9['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df9['h1433snWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df10= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_10 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_AIP1nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df10['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df10['AIP1nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df11= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_11 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_GADD45nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df11['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df11['GADD45nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df12= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_12 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_NOXAnWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df12['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df12['NOXAnWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df13= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_13 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_P53R2nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df13['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df13['P53R2nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df14= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Fortuno_2021_Kato_meta.xlsx", header = 0)
dc_14 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Fortuno_2021_Kato_meta.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Fortuno_2021_Kato_meta', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : 'NM_000546.5', 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df14['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df14['Fortuno_median'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df15= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_15 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_p53WT_Nutlin3', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df15["Position"],
               "aa_ref": df15["AA_wt"],
               "aa_alt": df15["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df15['A549_p53WT_Nutlin-3_Z-score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df16= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_16 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_p53null_Nutlin3', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df16["Position"],
               "aa_ref": df16["AA_wt"],
               "aa_alt": df16["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df16['A549_p53NULL_Nutlin-3_Z-score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df17= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_17 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_p53null_etoposide', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df17["Position"],
               "aa_ref": df17["AA_wt"],
               "aa_alt": df17["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df17['A549_p53NULL_Etoposide_Z-score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df18= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_18 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_combined_score', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df18["Position"],
               "aa_ref": df18["AA_wt"],
               "aa_alt": df18["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df18['Combined_score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#"Z" amino acids changed to "*", in Variant column where terminating amino acids are changed, Z is changed to "Ter", "B" is changed to "="
df19= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Fayer_2021.xlsx", header = 1)
dc_19 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Fayer_2021.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Fayer_2021_meta', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000546.5", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df19["Variant"],
               "consequence": np.nan,
               "auth_reported_score": df19['Classifier_prob_func_abnormal'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df19['Classifier_prediction'],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df20= pd.read_excel("~/Downloads/Pillar_project_data_files/PTEN_Mighell_2018.xlsx", header = 1)
dc_20 = {"x": "~/Downloads/Pillar_project_data_files/PTEN_Mighell_2018.xlsx", 
               "y": 1,
               "Dataset" :'PTEN_Mighell_2018', 
               "Gene" : "PTEN", 
               "HGNC_id" : 9588, 
               "Chrom" : 10, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000314.6", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df20["Variant (one letter)"],
               "consequence": np.nan,
               "auth_reported_score": df20['Cum_score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df20['High_conf'],
               "auth_reported_normal_min": -1.11,
               "auth_reported_normal_max": 0.89,
               "auth_reported_abnormal_min": -5.76,
               "auth_reported_abnormal_max":-2.13,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df21= pd.read_excel("~/Downloads/Pillar_project_data_files/PTEN_Matreyek_2018.xlsx", header = 0)
dc_21 = {"x": "~/Downloads/Pillar_project_data_files/PTEN_Matreyek_2018.xlsx", 
               "y": 0,
               "Dataset" :'PTEN_Matreyek_2018', 
               "Gene" : "PTEN", 
               "HGNC_id" : 9588, 
               "Chrom" : 10, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df21["hgvs_pro"],
               "consequence": np.nan,
               "auth_reported_score": df21['score'],
               "auth_reported_rep_score": df21[['score1','score2','score3','score4','score5',
                                                'score6','score7','score8']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": 0.71,
               "auth_reported_normal_max": 1.462,
               "auth_reported_abnormal_min": -0.223,
               "auth_reported_abnormal_max":0.4,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#changed "X" to "*", nonsense
df22= pd.read_excel("~/Downloads/Pillar_project_data_files/SCN5A_Glazer_2020.xlsx", header = 0)
dc_22 = {"x": "~/Downloads/Pillar_project_data_files/SCN5A_Glazer_2020.xlsx", 
               "y": 0,
               "Dataset" :'SCN5A_Glazer_2020', 
               "Gene" : "SCN5A", 
               "HGNC_id" : 10593, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "ENST00000333535", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df22["aa_num"],
               "aa_ref": df22["wt_allele"],
               "aa_alt": df22["mut_allele"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df22['dms'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df22["class"],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

df23= pd.read_excel("~/Downloads/Pillar_project_data_files/VHL_Buckley_2024.xlsx", header = 2)
dc_23 = {"x": "~/Downloads/Pillar_project_data_files/VHL_Buckley_2024.xlsx", 
               "y": 2,
               "Dataset" :'VHL_Buckley_2024', 
               "Gene" : "VHL", 
               "HGNC_id" : 12687, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df23["hg38_pos"], 
               "ref_allele": df23["ref"], 
               "alt_allele": df23["alt"], 
               "auth_transcript_id" : "ENST00000256474.3", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df23["protPos"],
               "aa_ref": df23["oAA"],
               "aa_alt": df23["nAA"],
               "hgvs_c": df23["cHGVS"],
               "hgvs_p" : df23["pHGVS"],
               "consequence": np.nan,
               "auth_reported_score": df23['function_score_final'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df23["function_class"],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'Yes'}

#mutAA column "X" replaced with "*"
df24= pd.read_excel("~/Downloads/Pillar_project_data_files/KCNH2_Kozek_Glazer_2020.xlsx", header = 1)
dc_24 = {"x": "~/Downloads/Pillar_project_data_files/KCNH2_Kozek_Glazer_2020.xlsx", 
               "y": 1,
               "Dataset" :'KCNH2_Kozek_Glazer_2020', 
               "Gene" : "KCNH2", 
               "HGNC_id" : 6251, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "ENST00000262186", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df24["resnum"],
               "aa_ref": df24["nativeAA"],
               "aa_alt": df24["mutAA"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df24['score.ave'],
               "auth_reported_rep_score": df24[['score.1', 'score.1']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": 75,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":75,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#had to reformat excel file to run through script; reformatting done in excel
df25= pd.read_excel("~/Downloads/Pillar_project_data_files/KCNH2_Jiang_2022.xlsx", header = 0)
dc_25 = {"x": "~/Downloads/Pillar_project_data_files/KCNH2_Jiang_2022.xlsx", 
               "y": 0,
               "Dataset" :'KCNH2_Jiang_2022', 
               "Gene" : "KCNH2", 
               "HGNC_id" : 6251, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000238.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df25["Protein Change"],
               "consequence": np.nan,
               "auth_reported_score": df25[' Normalised Current '],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": 0.78,
               "auth_reported_normal_max": 1.22,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":0.78,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#added 'p.' in the Human variant column
df26= pd.read_excel("~/Downloads/Pillar_project_data_files/OTC_Lo_2023.xlsx", header = 1)
dc_26 = {"x": "~/Downloads/Pillar_project_data_files/OTC_Lo_2023.xlsx", 
               "y": 1,
               "Dataset" :'OTC_Lo_2023', 
               "Gene" : "OTC", 
               "HGNC_id" : 8512, 
               "Chrom" : "X", 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000531.6", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df26["Human_Variant"],
               "consequence": np.nan,
               "auth_reported_score": df26['Growth Estimate'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df26['Functional_Class'],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#JAG1 supplement, added columns for aa_alt and aa_ref and functional class (according to author encoded functional class)
df27= pd.read_excel("~/Downloads/Pillar_project_data_files/JAG1_Gilbert_2024.xlsx", header = 1)
dc_27 = {"x": "~/Downloads/Pillar_project_data_files/JAG1_Gilbert_2024.xlsx", 
               "y": 1,
               "Dataset" :'JAG1_Gilbert_2024', 
               "Gene" : "JAG1", 
               "HGNC_id" : 6188, 
               "Chrom" : 20, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df27["pos"], 
               "ref_allele": df27["Ref_Allele"], 
               "alt_allele": df27["Alt_Allele"], 
               "auth_transcript_id" : "NM_000214.3", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df27["AA_Position"],
               "aa_ref": df27["aa_ref"],
               "aa_alt": df27["aa_alt"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": df27["Consequence"],
               "auth_reported_score": df27['meanAcrossReps'],
               "auth_reported_rep_score": df27[['VarScore_1', 'VarScore_2','VarScore_3','VarScore_4','VarScore_5',
                                               'VarScore_6','VarScore_7']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": df27['func_class'],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#BRCA2_Sahu_2023 supplement, new column added for hgvs_c without transcript annotation, "Intronic" removed from hgvs.p column
df28= pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", header = 0)
dc_28 = {"x": "~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_Sahu_2023_exon13_SGE', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df28["hgvs_c"],
               "hgvs_p" : df28["p.Nomenclature"],
               "consequence": np.nan,
               "auth_reported_score": df28['Function score DMSO'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": "nucleotide",
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

#BRCA2_Sahu_2023 supplement, new column added for hgvs_c without transcript annotation,"Intronic" removed from hgvs.p column
df29= pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", header = 0)
dc_29 = {"x": "~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_Sahu_2023_exon13_Cisplatin', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df29["hgvs_c"],
               "hgvs_p" : df29["p.Nomenclature"],
               "consequence": np.nan,
               "auth_reported_score": df29['Function score cisplatin'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

#BRCA2_Sahu_2023 supplement, new column added for hgvs_c without transcript annotation,"Intronic" removed from hgvs.p column
df30= pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", header = 0)
dc_30 = {"x": "~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_Sahu_2023_exon13_Olaparib', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df30["hgvs_c"],
               "hgvs_p" : df30["p.Nomenclature"],
               "consequence": np.nan,
               "auth_reported_score": df30['Function score olaparib'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

#assertions according to authors added as a new column in the excel sheet 
df31= pd.read_excel("~/Downloads/Pillar_project_data_files/SCN5A_Ma_2024_current_density.xlsx", header = 0)
dc_31 = {"x": "~/Downloads/Pillar_project_data_files/SCN5A_Ma_2024_current_density.xlsx", 
               "y": 0,
               "Dataset" :'SCN5A_Ma_2024_current_density', 
               "Gene" : "SCN5A", 
               "HGNC_id" : 10593, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000335.5", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df31["Cell line"],
               "consequence": np.nan,
               "auth_reported_score": df31['CD sqrtNORM(-120mV) Mean'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df31['class'],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#TSC2 library 1 (tuberin domain), unpublished IGVF (Fowler lab), updated amino acid numbering in new columns
df32 = pd.read_excel("~/Downloads/Pillar_project_data_files/TSC2_tuberin_unpublished.xlsx", header = 0)
dc_32 = {"x": "~/Downloads/Pillar_project_data_files/TSC2_tuberin_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'TSC2_tuberin_unpublished', 
               "Gene" : "TSC2", 
               "HGNC_id" : 12363, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df32['aa_pos_updated'],
               "aa_ref": df32['aa_ref_updated'],
               "aa_alt": df32['aa_alt_updated'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df32['average'],
               "auth_reported_rep_score": df32[['abrep2', 'abrep3']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#TSC2 library 2 (rap gap domain), unpublished IGVF (Fowler lab), updated amino acid numbering in new columns
df33 = pd.read_excel("~/Downloads/Pillar_project_data_files/TSC2_rapgap_unpublished.xlsx", header = 0)
dc_33 = {"x": "~/Downloads/Pillar_project_data_files/TSC2_rapgap_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'TSC2_rapgap_unpublished', 
               "Gene" : "TSC2", 
               "HGNC_id" : 12363, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df33['aa_pos_updated'],
               "aa_ref": df33['aa_ref_updated'],
               "aa_alt": df33['aa_alt_updated'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df33['average'],
               "auth_reported_rep_score": df33[['abrep1','abrep2', 'abrep3']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#G6PD unppublished (Fowler lab)
df34 = pd.read_excel("~/Downloads/Pillar_project_data_files/G6PD_unpublished.xlsx", header = 0)
dc_34 = {"x": "~/Downloads/Pillar_project_data_files/G6PD_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'G6PD_unpublished', 
               "Gene" : "G6PD", 
               "HGNC_id" : 4057, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df34["aaChanges"],
               "consequence": np.nan,
               "auth_reported_score": df34['average'],
               "auth_reported_rep_score": df34[['abrep1','abrep2', 'abrep3']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#BARD1_unpublished (BBI)
df35 = pd.read_excel("~/Downloads/Pillar_project_data_files/BARD1_unpublished.xlsx", header = 0)
dc_35 = {"x": "~/Downloads/Pillar_project_data_files/BARD1_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'BARD1_unpublished', 
               "Gene" : "BARD1", 
               "HGNC_id" : 952, 
               "Chrom" : 2, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df35['pos'], 
               "ref_allele": df35['ref'], 
               "alt_allele": df35['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df35['snv_score'],
               "auth_reported_rep_score": df35[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}
#PALB2_unpublished (BBI)
df36 = pd.read_excel("~/Downloads/Pillar_project_data_files/PALB2_unpublished.xlsx", header = 0)
dc_36 = {"x": "~/Downloads/Pillar_project_data_files/PALB2_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'PALB2_unpublished', 
               "Gene" : "PALB2", 
               "HGNC_id" : 26144, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df36['pos'], 
               "ref_allele": df36['ref'], 
               "alt_allele": df36['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df36['snv_score'],
               "auth_reported_rep_score": df36[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#XRCC2_unpublished (BBI)
df37 = pd.read_excel("~/Downloads/Pillar_project_data_files/XRCC2_unpublished.xlsx", header = 0)
dc_37 = {"x": "~/Downloads/Pillar_project_data_files/XRCC2_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'XRCC2_unpublished', 
               "Gene" : "XRCC2", 
               "HGNC_id" : 12829, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df37['pos'], 
               "ref_allele": df37['ref'], 
               "alt_allele": df37['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df37['snv_score'],
               "auth_reported_rep_score": df37[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#SFPQ_unpublished (BBI)
df38 = pd.read_excel("~/Downloads/Pillar_project_data_files/SFPQ_unpublished.xlsx", header = 0)
dc_38 = {"x": "~/Downloads/Pillar_project_data_files/SFPQ_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'SFPQ_unpublished', 
               "Gene" : "SFPQ", 
               "HGNC_id" : 10774, 
               "Chrom" : 1, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df38['pos'], 
               "ref_allele": df38['ref'], 
               "alt_allele": df38['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df38['snv_score'],
               "auth_reported_rep_score": df38[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#NBN_unpublished (BBI)
df39 = pd.read_excel("~/Downloads/Pillar_project_data_files/NBN_unpublished.xlsx", header = 0)
dc_39 = {"x": "~/Downloads/Pillar_project_data_files/NBN_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'NBN_unpublished', 
               "Gene" : "NBN", 
               "HGNC_id" : 7652, 
               "Chrom" : 8, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df39['pos'], 
               "ref_allele": df39['ref'], 
               "alt_allele": df39['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df39['snv_score'],
               "auth_reported_rep_score": df39[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}
#RAD51D_unpublished (BBI)
df40 = pd.read_excel("~/Downloads/Pillar_project_data_files/RAD51D_unpublished.xlsx", header = 0)
dc_40 = {"x": "~/Downloads/Pillar_project_data_files/RAD51D_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'RAD51D_unpublished', 
               "Gene" : "RAD51D", 
               "HGNC_id" : 9823, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df40['pos'], 
               "ref_allele": df40['ref'], 
               "alt_allele": df40['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df40['snv_score'],
               "auth_reported_rep_score": df40[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}
#CTCF_unpublished (BBI)
df41 = pd.read_excel("~/Downloads/Pillar_project_data_files/CTCF_unpublished.xlsx", header = 0)
dc_41 = {"x": "~/Downloads/Pillar_project_data_files/CTCF_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'CTCF_unpublished', 
               "Gene" : "CTCF", 
               "HGNC_id" : 13723, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df41['pos'], 
               "ref_allele": df41['ref'], 
               "alt_allele": df41['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df41['snv_score'],
               "auth_reported_rep_score": df41[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#BAP1_Waters_2024; does not need VEP input file, has everything annotated already
df42 = pd.read_excel("~/Downloads/Pillar_project_data_files/BAP1_Waters_2024.xlsx", header = 2)
dc_42 = {"x": "~/Downloads/Pillar_project_data_files/BAP1_Waters_2024.xlsx", 
               "y": 2,
               "Dataset" :'BAP1_Waters_2024', 
               "Gene" : "BAP1", 
               "HGNC_id" : 950, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df42['pos'], 
               "ref_allele": df42['ref'], 
               "alt_allele": df42['alt'], 
               "auth_transcript_id" : 'ENST00000460680.6', 
               "transcript_pos" : df42['CDS_position'],
               "transcript_ref" : df42['ref'], 
               "transcript_alt" : df42['alt'],
               "aa_pos": df42['protein_position'],
               "aa_ref": df42['ref_aa'],
               "aa_alt": df42['alt_aa'],
               "hgvs_c": df42['HGVSc'],
               "hgvs_p" : np.nan,
               "consequence": df42['vep_consequence'],
               "auth_reported_score": df42['functional_score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df42['functional_classification'],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#CALM1/2/3 yeast complementation assay, HGNC ID and other identifiers for CALM1 is used here
df43 = pd.read_excel("~/Downloads/Pillar_project_data_files/CALM1_2_3_Weile_2017.xlsx", header = 0)
dc_43 = {"x": "~/Downloads/Pillar_project_data_files/CALM1_2_3_Weile_2017.xlsx", 
               "y": 0,
               "Dataset" :'CALM1_CALM2_CALM3_Weile_2017', 
               "Gene" : "CALM1_2_3", 
               "HGNC_id" : 1442, 
               "Chrom" : 14, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" :np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df43['mut'],
               "consequence": np.nan,
               "auth_reported_score": df43['screen.score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#TPK1 yeast complementation assay 
df44 = pd.read_excel("~/Downloads/Pillar_project_data_files/TPK1_Weile_2017.xlsx", header = 0)
dc_44 = {"x": "~/Downloads/Pillar_project_data_files/TPK1_Weile_2017.xlsx", 
               "y": 0,
               "Dataset" :'TPK1_Weile_2017', 
               "Gene" : "TPK1", 
               "HGNC_id" : 17358, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" :np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df44['mut'],
               "consequence": np.nan,
               "auth_reported_score": df44['screen.score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#DDX3X SGE
df45 = pd.read_excel("~/Downloads/Pillar_project_data_files/DDX3X_Radford_2023_cLFC_day15.xlsx", header = 0)
dc_45 = {"x": "~/Downloads/Pillar_project_data_files/DDX3X_Radford_2023_cLFC_day15.xlsx", 
               "y": 0,
               "Dataset" :'DDX3X_Radford_2023_cLFC_day15', 
               "Gene" : "DDX3X", 
               "HGNC_id" : 16393, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : df45['VCF_position'], 
               "ref_allele": df45['VCF_Ref'], 
               "alt_allele": df45['VCF_Alt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : df45['CDS_position'],
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df45['Protein_position'],
               "aa_ref": df45['ref_aa'],
               "aa_alt": df45['alt_aa'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": df45['Consequence'],
               "auth_reported_score": df45['D15_combined_LFC'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df45['SGE_functional_classification'],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}


#HMBS ubiquitous isoform

df46 = pd.read_excel("~/Downloads/Pillar_project_data_files/HMBS_van_Loggerenberg_2023_ubquitous.xlsx", header = 0)
dc_46 = {"x": "~/Downloads/Pillar_project_data_files/HMBS_van_Loggerenberg_2023_ubquitous.xlsx", 
               "y": 0,
               "Dataset" :'HMBS_van_Loggerenberg_2023_ubquitous', 
               "Gene" : "HMBS", 
               "HGNC_id" : 4982, 
               "Chrom" : 11, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df46['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df46['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#HMBS erythroid

df47 = pd.read_excel("~/Downloads/Pillar_project_data_files/HMBS_van_Loggerenberg_2023_erythroid.xlsx", header = 0)
dc_47 = {"x": "~/Downloads/Pillar_project_data_files/HMBS_van_Loggerenberg_2023_erythroid.xlsx", 
               "y": 0,
               "Dataset" :'HMBS_van_Loggerenberg_2023_erythroid', 
               "Gene" : "HMBS", 
               "HGNC_id" : 4982, 
               "Chrom" : 11, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df47['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df47['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df48 = pd.read_excel("~/Downloads/Pillar_project_data_files/HMBS_van_Loggerenberg_2023_combined.xlsx", header = 0)
dc_48 = {"x": "~/Downloads/Pillar_project_data_files/HMBS_van_Loggerenberg_2023_combined.xlsx", 
               "y": 0,
               "Dataset" :'HMBS_van_Loggerenberg_2023_combined', 
               "Gene" : "HMBS", 
               "HGNC_id" : 4982, 
               "Chrom" : 11, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df48['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df48['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#added modified hgvs_c column to dataset, took out extra information from bottom
df49 = pd.read_excel("~/Downloads/Pillar_project_data_files/RHO_Wan_2019.xlsx", header = 0)
dc_49 = {"x": "~/Downloads/Pillar_project_data_files/RHO_Wan_2019.xlsx", 
               "y": 0,
               "Dataset" :'RHO_Wan_2019', 
               "Gene" : "RHO", 
               "HGNC_id" : 10012, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df49['hgvs_c'],
               "hgvs_p" : df49['Protein description (HGVS RHO_v001:p. or NP_000530.1:p.)'],
               "consequence": np.nan,
               "auth_reported_score": df49['NGS-based surface RHO surface assay, mean'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}


#Carboxylation-sensitive FIX-specific antibody score, X replaced with * for terminating aa

df50 = pd.read_excel("~/Downloads/Pillar_project_data_files/F9_Popp_2024.xlsx", header = 0)
dc_50 = {"x": "~/Downloads/Pillar_project_data_files/F9_Popp_2024.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2024_carboxy_F9_specific', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df50['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df50['Carboxylation-sensitive FIX-specific antibody score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df51 = pd.read_excel("~/Downloads/Pillar_project_data_files/F9_Popp_2024.xlsx", header = 0)
dc_51 = {"x": "~/Downloads/Pillar_project_data_files/F9_Popp_2024.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2024_heavy_chain', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df51['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df51['Heavy chain antibody score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df52 = pd.read_excel("~/Downloads/Pillar_project_data_files/F9_Popp_2024.xlsx", header = 0)
dc_52 = {"x": "~/Downloads/Pillar_project_data_files/F9_Popp_2024.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2024_light_chain', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df52['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df52['Light chain antibody score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df53 = pd.read_excel("~/Downloads/Pillar_project_data_files/F9_Popp_2024.xlsx", header = 0)
dc_53 = {"x": "~/Downloads/Pillar_project_data_files/F9_Popp_2024.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2024_carboxy_gla_motif', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df53['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df53['Carboxylation-sensitive Gla-motif antibody score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df54 = pd.read_excel("~/Downloads/Pillar_project_data_files/F9_Popp_2024.xlsx", header = 0)
dc_54 = {"x": "~/Downloads/Pillar_project_data_files/F9_Popp_2024.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2024_strep_2', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df54['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df54['Strep II tag antibody score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

 
dc_list = [dc_1, dc_2, dc_3, dc_4,dc_5,dc_6,dc_7,dc_8,dc_9,dc_10,dc_11,dc_12,dc_13,dc_14,dc_15,
           dc_16,dc_17,dc_18,dc_19,dc_20,dc_21,dc_22,dc_23, dc_24,dc_25,dc_26,dc_27,dc_28,dc_29,dc_30,
          dc_31,dc_32,dc_33,dc_34,dc_35,dc_36,dc_37,dc_38,dc_39,dc_40,dc_41,dc_42,dc_43,dc_44,dc_45,dc_46,
          dc_47, dc_48,dc_49,dc_50,dc_51,dc_52,dc_53,dc_54]

final_combined_df = data_harmonization_loop(dc_list)

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:312: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


KeyError for row 3789 with value _wt: 't', ~/Downloads/Pillar_project_data_files/PTEN_Matreyek_2018.xlsx
KeyError for row 10677 with value silent: 's', ~/Downloads/Pillar_project_data_files/G6PD_unpublished.xlsx
KeyError for row 10677 with value silent: 'len', ~/Downloads/Pillar_project_data_files/G6PD_unpublished.xlsx
KeyError for row 131 with value *349Qext*51: '1', ~/Downloads/Pillar_project_data_files/RHO_Wan_2019.xlsx
KeyError for row 132 with value *349Eext*51: '1', ~/Downloads/Pillar_project_data_files/RHO_Wan_2019.xlsx
KeyError for row 133 with value synonymous/splice: 's', ~/Downloads/Pillar_project_data_files/RHO_Wan_2019.xlsx
KeyError for row 144 with value I256del: 'l', ~/Downloads/Pillar_project_data_files/RHO_Wan_2019.xlsx
KeyError for row 145 with value P327Wfs*32: '2', ~/Downloads/Pillar_project_data_files/RHO_Wan_2019.xlsx
KeyError for row 146 with value R69_L72del: 'l', ~/Downloads/Pillar_project_data_files/RHO_Wan_2019.xlsx
KeyError for row 147 with value P327Hfs*33:

In [127]:
#only for use when trying to increase the dataframe incrementally

# full_df = pd.read_csv("~/Downloads/pillar_data_combined_df_v4.csv")


# #function to add dataframes one at time after
# def incremental_data_harmonization(existing_df, dc):
    
#     # Harmonize the new data chunk
#     harmonized_df = data_harmonization(dc)
    
#     # Append the new harmonized data to the existing dataframe
#     if harmonized_df is not None and not harmonized_df.empty:
#         final_combined_df = pd.concat([existing_df, harmonized_df], ignore_index=True)
    
#     return final_combined_df


# final_combined_df = incremental_data_harmonization(full_df,dc_54)

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (3,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [163]:
final_combined_df

,Dataset,Gene,HGNC_id,Chrom,hg19_pos,hg38_start,ref_allele,alt_allele,auth_transcript_id,transcript_pos,transcript_ref,transcript_alt,aa_pos,aa_ref,aa_alt,hgvs_c,hgvs_p,consequence,auth_reported_score,auth_reported_rep_score,auth_reported_func_class,auth_reported_normal_min,auth_reported_normal_max,auth_reported_abnormal_min,auth_reported_abnormal_max,splice_measure,gnomad_MAF,clinvar_sig,clinvar_star,clinvar_date_last_reviewed,nucleotide_or_aa
0,BRCA1_Findlay_2018,BRCA1,1100,17,41276135.0,NaN,T,G,NM_007294.3,-19-3,A,C,NaN,NaN,NaN,c.-19-3A>C,NaN,Splice region,-0.372611,-0.568675758504309;-0.176545248348852,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,NaN,NaN,NaN,nucleotide
1,BRCA1_Findlay_2018,BRCA1,1100,17,41276135.0,NaN,T,C,NM_007294.3,-19-3,A,G,NaN,NaN,NaN,c.-19-3A>G,NaN,Splice region,-0.045313,-0.332667114078832;0.242040175629815,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,NaN,NaN,NaN,nucleotide
2,BRCA1_Findlay_2018,BRCA1,1100,17,41276135.0,NaN,T,A,NM_007294.3,-19-3,A,T,NaN,NaN,NaN,c.-19-3A>T,NaN,Splice region,-0.108254,-0.440230467981572;0.223722191334123,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,NaN,NaN,NaN,nucleotide
3,BRCA1_Findlay_2018,BRCA1,1100,17,41276134.0,NaN,T,G,NM_007294.3,-19-2,A,C,NaN,NaN,NaN,c.-19-2A>C,NaN,Canonical splice,-0.277963,-0.41057727764695;-0.145348986773218,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,NaN,NaN,NaN,nucleotide
4,BRCA1_Findlay_2018,BRCA1,1100,17,41276134.0,NaN,T,C,NM_007294.3,-19-2,A,G,NaN,NaN,NaN,c.-19-2A>G,NaN,Canonical splice,-0.388414,-0.6350191556350679;-0.141808664620659,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,NaN,NaN,NaN,nucleotide
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269489,F9_Popp_2024_strep_2,F9,3551,X,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,461,T,H,NaN,p.Thr461His,NaN,1.216781,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,aa
269490,F9_Popp_2024_strep_2,F9,3551,X,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,461,T,K,NaN,p.Thr461Lys,NaN,1.129666,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,aa
269491,F9_Popp_2024_strep_2,F9,3551,X,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,461,T,D,NaN,p.Thr461Asp,NaN,1.180239,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,aa
269492,F9_Popp_2024_strep_2,F9,3551,X,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,461,T,E,NaN,p.Thr461Glu,NaN,1.072726,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,aa


In [164]:
final_combined_df['ID'] = final_combined_df.apply(lambda row: f"{row['Dataset']}_var{row.name + 1}", axis=1)

final_combined_df.to_csv('~/Downloads/pillar_data_combined_df_v4.csv', index=False)

In [165]:
print(len(final_combined_df['ID'].unique()))

269494


In [166]:
import pandas as pd

p_data = pd.read_csv("~/Downloads/pillar_data_combined_df_v4.csv")

#some datasets need to have their ref/alt and transcript ref/alt set to the same allele given they are on
#the positive strand

condition = (p_data['Dataset'] == 'BRCA2_Hu_2024') | (p_data['Dataset'] == 'BRCA2_Sahu_2023_exon13_SGE') | (p_data['Dataset'] == 'BRCA2_Sahu_2023_exon13_Olaparib') | (p_data['Dataset'] == 'BRCA2_Sahu_2023_exon13_Cisplatin')
    
#setting ref/alt and transcript ref/alt to the same allele
p_data.loc[condition,'ref_allele'] = p_data.loc[condition,'transcript_ref']

p_data.loc[condition,'alt_allele'] = p_data.loc[condition,'transcript_alt']

#load big curation sheet for merge
curation_data = pd.read_csv("~/Downloads/MAVE_curation.csv", header = 1)

#merge and add additional information from the big curation sheet
df_merge = pd.merge(p_data, curation_data[['Dataset_tag', 'MaveDB URN', 'Ensembl_transript_ID', 
                                           "Ref_seq_transcript_ID","Model_system","Assay_type",
                                          "Phenotype_measured","Phenotype_detail","IGVF_produced"]], 
                                           left_on='Dataset', right_on = "Dataset_tag", how='left')
df_merge = df_merge.drop_duplicates()

df_merge.to_csv("~/Downloads/pillar_data_with_curation_v4.csv", index = False)

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (3,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [167]:
print(len(df_merge['ID'].unique()))

269494


In [168]:
import pandas as pd
import re

#load concatenated data 
pd_1 = pd.read_csv("~/Downloads/pillar_data_with_curation_v4.csv", header=0)

# Define the amino acid to codon dictionary
amino_acid_to_codon = {
    'F': ['TTT', 'TTC'],
    'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
    'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
    'Y': ['TAT', 'TAC'],
    'Ter': ['TAA', 'TAG', 'TGA'],
    '*': ['TAA', 'TAG', 'TGA'],
    'C': ['TGT', 'TGC'],
    'W': ['TGG'],
    'P': ['CCT', 'CCC', 'CCA', 'CCG'],
    'H': ['CAT', 'CAC'],
    'Q': ['CAA', 'CAG'],
    'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'I': ['ATT', 'ATC', 'ATA'],
    'M': ['ATG'],
    'T': ['ACT', 'ACC', 'ACA', 'ACG'],
    'N': ['AAT', 'AAC'],
    'K': ['AAA', 'AAG'],
    'V': ['GTT', 'GTC', 'GTA', 'GTG'],
    'A': ['GCT', 'GCC', 'GCA', 'GCG'],
    'D': ['GAT', 'GAC'],
    'E': ['GAA', 'GAG'],
    'G': ['GGT', 'GGC', 'GGA', 'GGG'],
    '-': ['del']
}

# Specify datasets that need the chromosome-based key generation approach for a downstream merge with VEP data
datasets_with_chromosome_key = {"BARD1_unpublished", "CTCF_unpublished", 'NBN_unpublished','BARD1_unpublished', 
                                'PALB2_unpublished','RAD51D_unpublished','SFPQ_unpublished','XRCC2_unpublished',
                               'BAP1_Waters_2024','DDX3X_Radford_2023_cLFC_day15'}

cdot_datasets = ['BRCA1_Findlay_2018',"VHL_Buckley_2024","BRCA2_Sahu_2023_exon13_SGE",
                 "BRCA2_Sahu_2023_exon13_Cisplatin","BRCA2_Sahu_2023_exon13_Olaparib","RHO_Wan_2019"]


expanded_rows = []

expanded_rows = []

for dataset, group in pd_1.groupby('Dataset'):
    # Check if the dataset requires chromosome-based notation
    if dataset in datasets_with_chromosome_key:
        for i, row in group.iterrows():
            try:
                chrom = row['Chrom']
                pos = int(row['hg38_start'])  
                ref = row['ref_allele']
                alt = row['alt_allele']

                # Handle chromosome formatting
                if str(chrom).upper() == "X":
                    chrom_str = "X"
                elif str(chrom).upper() == "Y":
                    chrom_str = "Y"
                else:
                    chrom = int(chrom)
                    chrom_str = f"{chrom:02}" if chrom < 10 else str(chrom)

                notation = f"NC_0000{chrom_str}:g.{pos}{ref}>{alt}"

            except (KeyError, ValueError) as e:
                notation = "placeholder_key"  # Placeholder when a key cannot be generated
                print(f"Error for row {i} in chromosome-based dataset: {e}")

            # Append the row with the generated or placeholder key
            expanded_row = row.to_dict()
            expanded_row['key'] = notation
            expanded_rows.append(expanded_row)

    elif dataset in cdot_datasets:
        for i, row in group.iterrows():
            try:
                hgvs_c = row['hgvs_c']
                ID = row['Ensembl_transript_ID']
                transcript = re.sub(r"(ENST\d+)\.\d+", r"\1", ID)

                key = f"{transcript}:{hgvs_c}"

            except (KeyError, ValueError) as e:
                key = "placeholder_key"  # Placeholder when a key cannot be generated
                print(f"Error for row {i} in CDOT dataset: {e}")

            # Append the row with the generated or placeholder key
            expanded_row = row.to_dict()
            expanded_row['key'] = key
            expanded_rows.append(expanded_row)

    else:
        for i, row in group.iterrows():
            try:
                value = row['aa_alt']
                pos = row['aa_pos']
                ID = row['Ensembl_transript_ID']
                aa_ref = row['aa_ref']

                amino_acid = aa_ref if value in ['0', '='] else value

                if (
                    pd.notna(ID)
                    and isinstance(ID, str)
                    and amino_acid in amino_acid_to_codon
                ):
                    transcript = re.sub(r"(ENST\d+)\.\d+", r"\1", ID)
                    codons = amino_acid_to_codon[amino_acid]
                    nucleotide_pos = (int(float(pos)) * 3) - 2

                    for codon in codons:
                        if amino_acid == '-':
                            key = f"{transcript}:c.{int(nucleotide_pos)}_{int(nucleotide_pos + 2)}{codon}"
                        else:
                            key = f"{transcript}:c.{int(nucleotide_pos)}_{int(nucleotide_pos + 2)}delins{codon}"

                        expanded_row = row.to_dict()
                        expanded_row['key'] = key
                        expanded_rows.append(expanded_row)

                else:
                    raise ValueError("Invalid amino acid data")

            except (KeyError, ValueError) as e:
                key = "placeholder_key" 
                dataset = row['Dataset']
                print(f"Error for row {i} {dataset} : {value}{pos}{ID} in amino_acid-based dataset: {e}")

                # Append the row with the placeholder key
                expanded_row = row.to_dict()
                expanded_row['key'] = key
                expanded_rows.append(expanded_row)

# Convert expanded rows into a new DataFrame
expanded_df = pd.DataFrame(expanded_rows)

# Save to CSV
expanded_df.to_csv("~/Downloads/pillar_data_with_curation_with_key_v4.csv", index=False)

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (3,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,33,35,36,38,39,40) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


Error for row 129061 G6PD_unpublished : TnanENST00000393562.10 in amino_acid-based dataset: cannot convert float NaN to integer
Error for row 129504 G6PD_unpublished : silentnanENST00000393562.10 in amino_acid-based dataset: Invalid amino acid data
Error for row 100205 PTEN_Matreyek_2018 : _wtnanENST00000371953.8 in amino_acid-based dataset: Invalid amino acid data
Error for row 108810 SCN5A_Ma_2024_current_density : ACN5ENST00000423572.7 in amino_acid-based dataset: could not convert string to float: 'CN5'


In [198]:
print(len(expanded_df['ID'].unique()))

269494


In [1]:
import pandas as pd

#load pillar project combined data with key gen
pd_1_key = pd.read_csv("~/Downloads/pillar_data_with_curation_with_key_v4.csv")

#strip white spaces and such 
pd_1_key = pd_1_key.applymap(lambda x: x.strip() if isinstance(x, str) else x)

#load combined VEP output
pd_2_key = pd.read_csv("~/Downloads/VEP_output_v4.csv")

#strip white spaces and such
pd_2_key = pd_2_key.applymap(lambda x: x.strip() if isinstance(x, str) else x)

pd_2_key = pd_2_key.apply(pd.to_numeric, errors='ignore')

#define datatypes for columns to be merged 
pd_1_key['key'] = pd_1_key['key'].astype(str)
pd_1_key['Ensembl_transript_ID'] = pd_1_key['Ensembl_transript_ID'].astype(str)
pd_2_key['#Uploaded_variation'] = pd_2_key['#Uploaded_variation'].astype(str)
pd_2_key['Feature'] = pd_2_key['Feature'].astype(str)

#filter on datasets with only amino acid information first to carry through for some changes downstream
pd_1_filtered = pd_1_key[pd_1_key['nucleotide_or_aa'] == 'aa']

#merge on key and ensembl transcript ID 
pd_merged = pd.merge(pd_1_filtered, pd_2_key[["Location","Allele","Consequence","HGVSc","HGVSp",
                                          "cDNA_position","CDS_position","Protein_position","Amino_acids",
                                          "Codons","REF_ALLELE","Feature","#Uploaded_variation","UPLOADED_ALLELE",
                                          "STRAND"]], 
                                          left_on=["key",'Ensembl_transript_ID'],
                                          right_on = ["#Uploaded_variation","Feature"],how='left')

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (3,6,7,8,9,10,11,12,15,17,18,19,20,33,35,36,37,38,39,40) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (16,23,30) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [2]:
#take the merged dataframe and clean it up and drop any duplicates

pd_merged = pd_merged.applymap(lambda x: x.strip() if isinstance(x, str) else x)

pd_merged = pd_merged.apply(pd.to_numeric, errors='ignore')

pd_merged_clean = pd_merged.drop_duplicates()

In [3]:
import re

nuc_dic = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}

def translate_sequence(sequence):
    return ''.join(nuc_dic.get(nuc, nuc) for nuc in sequence)[::-1]

def process_alleles(row):
    if pd.notna(row['UPLOADED_ALLELE']):
        ref, alt = str(row['UPLOADED_ALLELE']).split("/")
        if row['STRAND'] == -1:
            return translate_sequence(ref), translate_sequence(alt)
        return ref, alt
    return row['UPLOADED_ALLELE'], row['UPLOADED_ALLELE']

pd_merged_clean[['ref_allele', 'alt_allele']] = pd_merged_clean.apply(process_alleles, axis=1, result_type="expand")

def extract_hg38_position(value):
    if pd.notna(value):
        chrom, pos = str(value).split(":")
        start, end = pos.split("-")
        return start, end
    return value, value

pd_merged_clean[['hg38_start', 'hg38_end']] = pd_merged_clean['Location'].apply(extract_hg38_position).apply(pd.Series)

pd_merged_clean['hgvs_c'] = pd_merged_clean['HGVSc'].apply(lambda x: x.split(":")[1] if pd.notna(x) and ":" in x else x)

pd_merged_clean['transcript_pos'] = pd_merged_clean['CDS_position']
pd_merged_clean['consequence'] = pd_merged_clean['Consequence']


/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/pandas/core/frame.py:3641: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self[k1] = value[k2]
/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1420/363544896.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_merged_clean['hgvs_c'] = pd_merged_clean['HGVSc'].apply(lambda x: x.split(":")[1] if pd.notna(x) and ":" in x else x)
/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1420/363544896.py:29: SettingWithCopyWarning: 
A

In [4]:
pd_1_key_aa = pd_1_key[pd_1_key['nucleotide_or_aa'] == 'aa']

final_df = pd.merge(pd_1_key_aa, pd_merged_clean, on=['Dataset', 'key'], how='left', suffixes=('', '_drop'))



columns_to_update = ['hg38_start', 'ref_allele', 'alt_allele','transcript_pos',
       'transcript_ref', 'transcript_alt','consequence']
columns_with_fallback = ['hg38_start_drop', 'ref_allele_drop', 'alt_allele_drop',
                        'transcript_pos_drop', 'transcript_ref_drop',
                        'transcript_alt_drop','consequence_drop']

for x_col, y_col in zip(columns_to_update, columns_with_fallback):
    final_df[x_col] = final_df[x_col].fillna(final_df[y_col])  

final_df

final_df = final_df[[col for col in final_df.columns if not col.endswith('_drop')]]

final_df.drop(columns=['Dataset_tag','key','Location', 'Allele', 'Consequence', 'HGVSc',
       'HGVSp', 'cDNA_position', 'CDS_position', 'Protein_position',
       'Amino_acids', 'Codons', 'REF_ALLELE', 'Feature', '#Uploaded_variation',
       'UPLOADED_ALLELE'], inplace=True)

new_column_order = ['ID','Dataset', 'Gene', 'HGNC_id', 'Chrom','STRAND','hg19_pos', 'hg38_start','hg38_end',
       'ref_allele', 'alt_allele', 'auth_transcript_id', 'transcript_pos',
       'transcript_ref', 'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt',
       'hgvs_c', 'hgvs_p', 'consequence', 'auth_reported_score',
       'auth_reported_rep_score', 'auth_reported_func_class',
       'auth_reported_normal_min', 'auth_reported_normal_max',
       'auth_reported_abnormal_min', 'auth_reported_abnormal_max',
       'splice_measure', 'gnomad_MAF', 'clinvar_sig', 'clinvar_star',
       'clinvar_date_last_reviewed', 'nucleotide_or_aa', 'MaveDB URN',
       'Ensembl_transript_ID', 'Ref_seq_transcript_ID', 'Model_system',
       'Assay_type', 'Phenotype_measured', 'Phenotype_detail', 'IGVF_produced']

final_df = final_df[new_column_order]

final_df_2 = final_df.drop_duplicates()

In [5]:
pd_1 = pd.read_csv("~/Downloads/pillar_data_with_curation_v4.csv", header=0)

pd_1.drop(columns=['Dataset_tag'], inplace=True)

pd_1_nuc = pd_1[pd_1['nucleotide_or_aa'] == 'nucleotide']

final_df_3 = pd.concat([final_df_2, pd_1_nuc], axis=0, ignore_index=True)

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (3,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,33,35,36,38,39,40) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
final_df_3

,ID,Dataset,Gene,HGNC_id,Chrom,STRAND,hg19_pos,hg38_start,hg38_end,ref_allele,...,clinvar_date_last_reviewed,nucleotide_or_aa,MaveDB URN,Ensembl_transript_ID,Ref_seq_transcript_ID,Model_system,Assay_type,Phenotype_measured,Phenotype_detail,IGVF_produced
0,BRCA1_Adamovich_2022_Cisplatin_var6165,BRCA1_Adamovich_2022_Cisplatin,BRCA1,1100,17,-1.0,NaN,43067663,43067665,GTG,...,NaN,aa,urn:mavedb:00001209-a-2,ENST00000357654.9,NM_007294.4,immortalized human cells,Cell Viability,Drug Resisance,Cisplatin resistance,No
1,BRCA1_Adamovich_2022_Cisplatin_var6165,BRCA1_Adamovich_2022_Cisplatin,BRCA1,1100,17,-1.0,NaN,43067663,43067664,TG,...,NaN,aa,urn:mavedb:00001209-a-2,ENST00000357654.9,NM_007294.4,immortalized human cells,Cell Viability,Drug Resisance,Cisplatin resistance,No
2,BRCA1_Adamovich_2022_Cisplatin_var6166,BRCA1_Adamovich_2022_Cisplatin,BRCA1,1100,17,-1.0,NaN,43067663,43067665,GTG,...,NaN,aa,urn:mavedb:00001209-a-2,ENST00000357654.9,NM_007294.4,immortalized human cells,Cell Viability,Drug Resisance,Cisplatin resistance,No
3,BRCA1_Adamovich_2022_Cisplatin_var6166,BRCA1_Adamovich_2022_Cisplatin,BRCA1,1100,17,-1.0,NaN,43067663,43067665,GTG,...,NaN,aa,urn:mavedb:00001209-a-2,ENST00000357654.9,NM_007294.4,immortalized human cells,Cell Viability,Drug Resisance,Cisplatin resistance,No
4,BRCA1_Adamovich_2022_Cisplatin_var6166,BRCA1_Adamovich_2022_Cisplatin,BRCA1,1100,17,-1.0,NaN,43067664,43067665,GT,...,NaN,aa,urn:mavedb:00001209-a-2,ENST00000357654.9,NM_007294.4,immortalized human cells,Cell Viability,Drug Resisance,Cisplatin resistance,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
680410,RHO_Wan_2019_var221085,RHO_Wan_2019,RHO,10012,3,NaN,NaN,NaN,NaN,NaN,...,NaN,nucleotide,urn:mavedb:00000099,ENST00000296271.4,NM_000539.3,immortalized human cells,Reporter,Fluorescence,cell surface expression,No
680411,RHO_Wan_2019_var221086,RHO_Wan_2019,RHO,10012,3,NaN,NaN,NaN,NaN,NaN,...,NaN,nucleotide,urn:mavedb:00000099,ENST00000296271.4,NM_000539.3,immortalized human cells,Reporter,Fluorescence,cell surface expression,No
680412,RHO_Wan_2019_var221087,RHO_Wan_2019,RHO,10012,3,NaN,NaN,NaN,NaN,NaN,...,NaN,nucleotide,urn:mavedb:00000099,ENST00000296271.4,NM_000539.3,immortalized human cells,Reporter,Fluorescence,cell surface expression,No
680413,RHO_Wan_2019_var221088,RHO_Wan_2019,RHO,10012,3,NaN,NaN,NaN,NaN,NaN,...,NaN,nucleotide,urn:mavedb:00000099,ENST00000296271.4,NM_000539.3,immortalized human cells,Reporter,Fluorescence,cell surface expression,No


In [7]:
print(len(final_df_3['ID'].unique()))

269494


In [8]:
final_df_3.to_csv("~/Downloads/final_pillar_data_v4.csv", index = False)

In [208]:

final_df_3 = pd.read_csv("~/Downloads/final_pillar_data_v4.csv")


# #nucleotide datasets need some special adjustments to get everything in the right format 

# #BRCA1_2018_Findlay needs hg38 positions annotated

import pandas as pd
from pyliftover import LiftOver

lo = LiftOver('/Users/malvikatejura/Downloads/hg19ToHg38.over.chain.gz')

# Define a function to lift over positions
def lift_over_pos(row):
    chrom = f"chr{row['Chrom']}"
    pos = row['hg19_pos']
    result = lo.convert_coordinate(chrom, pos)
    if result:
        return result[0][1] 
    else:
        return None 

final_df_3['hg38_start'] = final_df_3.apply(
    lambda row: lift_over_pos(row) if row['Dataset'] == 'BRCA1_Findlay_2018' else row['hg38_start'],
    axis=1
)

#JAG1_Gilbert needs transcript information and hgvs_c annotated (already has hg38 positions)


/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (4,11,12,13,14,15,16,17,18,19,20,21,22,23,34,37,38,39,40,41) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [9]:
#all new SGE data needs auth transcript annotated, amino acid information, transcript pos and other information, 
#hgvs_c, hgvs_p,consequence

datasets_with_chromosome_key = {"BARD1_unpublished", "CTCF_unpublished", 'NBN_unpublished','BARD1_unpublished', 
                                'PALB2_unpublished','RAD51D_unpublished','SFPQ_unpublished','XRCC2_unpublished'}


transcripts_for_filter = {"ENST00000260947.9", "ENST00000264010.10", "ENST00000345365.11", "ENST00000261584.9",
"ENST00000357214.6","ENST00000265433.8","ENST00000359321.2"}



pd_2_key_chr = pd_2_key[(pd_2_key['Feature'].isin(transcripts_for_filter))] 


pd_2_key_chr['hg38_start'] = pd_2_key_chr['#Uploaded_variation'].str.extract(r'g\.(\d+)')
pd_2_key_chr['ref_allele'] = pd_2_key_chr['#Uploaded_variation'].str.extract(r'g\.\d+([A-Z])>')
pd_2_key_chr['alt_allele'] = pd_2_key_chr['#Uploaded_variation'].str.extract(r'>([A-Z])')
pd_2_key_chr['hgvs_c'] = pd_2_key_chr['HGVSc'].str.extract(r':(.*)')
pd_2_key_chr['transcript_pos'] = pd_2_key_chr['HGVSc'].str.extract(r'c\.([\d+-]+)')
pd_2_key_chr['transcript_ref'] = pd_2_key_chr['HGVSc'].str.extract(r'[+-]?\d+([A-Z])>')
pd_2_key_chr['transcript_alt'] = pd_2_key_chr['HGVSc'].str.extract(r'>([A-Z])')
pd_2_key_chr['hgvs_p'] = pd_2_key_chr['HGVSp'].str.extract(r':(p\.\w+\d+\w+)')
pd_2_key_chr['aa_pos'] = pd_2_key_chr['Protein_position']
pd_2_key_chr[['aa_ref', 'aa_alt']] = pd_2_key_chr['Amino_acids'].str.split('/', expand=True)
pd_2_key_chr['consequence'] = pd_2_key_chr['Consequence']


# Convert hg38_start to integer
pd_2_key_chr['hg38_start'] = pd_2_key_chr['hg38_start'].astype('float64')
final_df_3['hg38_start'] = final_df_3['hg38_start'].astype('float64')



merged_df = final_df_3.merge(
    pd_2_key_chr[[
        'hg38_start', 'ref_allele', 'alt_allele', 'hgvs_c',
        'transcript_pos', 'transcript_ref', 'transcript_alt', 
        'hgvs_p', 'aa_pos', 'aa_ref', 'aa_alt','consequence'
    ]],
    on=['hg38_start', 'ref_allele', 'alt_allele'],
    how='left', 
    suffixes=('', '_new')
)

# Ensure the merged DataFrame uses the same index as final_df_3
merged_df.index = final_df_3.index

# Update the relevant columns only for rows in the specified datasets
for col in ['hgvs_c', 'transcript_pos', 'transcript_ref', 'transcript_alt', 'hgvs_p', 'aa_pos', 'aa_ref', 'aa_alt','consequence']:
    final_df_3.loc[final_df_3['Dataset'].isin(datasets_with_chromosome_key), col] = (
        merged_df.loc[final_df_3['Dataset'].isin(datasets_with_chromosome_key), f'{col}_new']
    )
    
final_df_3.loc[final_df_3['Dataset'].isin(datasets_with_chromosome_key), 'hg38_end'] = (
    final_df_3.loc[final_df_3['Dataset'].isin(datasets_with_chromosome_key), 'hg38_start']
)

#BAP1 needs hgvs_c and hgvs_p normalized to match the dataframe

final_df_3.loc[final_df_3['Dataset'] == 'BAP1_Waters_2024', 'hgvs_c'] = final_df_3.loc[
    final_df_3['Dataset'] == 'BAP1_Waters_2024', 'hgvs_c'
].str.split(':').str[1]


/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1420/12619565.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_2_key_chr['hg38_start'] = pd_2_key_chr['#Uploaded_variation'].str.extract(r'g\.(\d+)')
/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1420/12619565.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_2_key_chr['ref_allele'] = pd_2_key_chr['#Uploaded_variation'].str.extract(r'g\.\d+([A-Z])>')
/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1420/12619565.p

In [10]:
final_df_3.to_csv("~/Downloads/final_pillar_data_v5.csv", index = False)

In [11]:
#BRCA2_Hu and #BRCA2_sahu(all) need hg38 positions annotated and consequence

def extract_hg38_position(value):
    if pd.notna(value):
        chrom, pos = str(value).split(":")
        start, end = pos.split("-")
        return start, end
    return value, value

pd_2_key_BRCA2 = pd_2_key[(pd_2_key['SYMBOL'] == 'BRCA2') & (pd_2_key['Feature'] == 'ENST00000380152.8')]

pd_2_key_BRCA2['hgvs_c'] = pd_2_key['HGVSc'].str.extract(r':(.*)')

pd_2_key_BRCA2[['aa_ref', 'aa_alt']] = pd_2_key['Amino_acids'].str.split('/', expand=True)

pd_2_key_BRCA2['aa_pos'] = pd_2_key['Protein_position']

# Define the datasets to update
datasets_x = {"BRCA2_Hu_2024","BRCA2_Sahu_2023_exon13_Cisplatin","BRCA2_Sahu_2023_exon13_Olaparib",
            "BRCA2_Sahu_2023_exon13_SGE"}

key_mapping_c = pd_2_key_BRCA2.set_index('hgvs_c')[['Location', 'Consequence']].to_dict('index')

# Function to update positions and consequence
def update_positions_and_consequence(row):
    if row['Dataset'] in datasets_x:  
        mapping = key_mapping_c.get(row['hgvs_c'])
        
#         # Fallback to amino acid keys if hgvs_c is unavailable
#         if pd.isna(mapping) or row['hgvs_c'] in [None, '']:
#             mapping = key_mapping_aa.get((row['aa_ref'], row['aa_alt'], row['aa_pos']))
        
        if mapping:  # If a mapping is found
            location = mapping['Location']
            consequence = mapping['Consequence']
            hg38_start_new, hg38_end_new = extract_hg38_position(location)
            return (
                hg38_start_new or row['hg38_start'], 
                hg38_end_new or row['hg38_end'], 
                consequence or row['consequence']
            )
    
    # If no update, return original values
    return row['hg38_start'], row['hg38_end'], row['consequence']

# Apply the function to update hg38_start, hg38_end, and consequence
final_df_3[['hg38_start', 'hg38_end', 'consequence']] = final_df_3.apply(
    lambda row: pd.Series(update_positions_and_consequence(row)),
    axis=1
)

/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1420/2074478687.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_2_key_BRCA2['hgvs_c'] = pd_2_key['HGVSc'].str.extract(r':(.*)')
/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1420/2074478687.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_2_key_BRCA2['aa_pos'] = pd_2_key['Protein_position']


In [12]:
print(len(final_df_3['ID'].unique()))

269494


In [13]:
final_df_3.to_csv("~/Downloads/final_pillar_data_v6.csv", index = False)

In [29]:
final_df_3 = pd.read_csv("~/Downloads/final_pillar_data_v6.csv")

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (4,11,13,14,15,18,21,22,23,34,37,38,39,40,41) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [30]:
#RHO needs consequence, hg38 positions and strand info updated

pd_2_key_RHO = pd_2_key[(pd_2_key['SYMBOL'] == 'RHO') & (pd_2_key['Feature'] == 'ENST00000296271.4')]

pd_2_key_RHO['hgvs_c'] = pd_2_key['HGVSc'].str.extract(r':(.*)')

# Define the datasets to update
datasets_R = {"RHO_Wan_2019"}

key_mapping_r = pd_2_key_RHO.set_index('hgvs_c')[['Location', 'Consequence']].to_dict('index')

# Function to update positions and consequence
def update_positions_and_consequence(row):
    if row['Dataset'] in datasets_R:  
        mapping = key_mapping_r.get(row['hgvs_c'])
        
#         # Fallback to amino acid keys if hgvs_c is unavailable
#         if pd.isna(mapping) or row['hgvs_c'] in [None, '']:
#             mapping = key_mapping_aa.get((row['aa_ref'], row['aa_alt'], row['aa_pos']))
        
        if mapping:  # If a mapping is found
            location = mapping['Location']
            consequence = mapping['Consequence']
            hg38_start_new, hg38_end_new = extract_hg38_position(location)
            return (
                hg38_start_new or row['hg38_start'], 
                hg38_end_new or row['hg38_end'], 
                consequence or row['consequence']
            )
    
    # If no update, return original values
    return row['hg38_start'], row['hg38_end'], row['consequence']

# Apply the function to update hg38_start, hg38_end, and consequence
final_df_3[['hg38_start', 'hg38_end', 'consequence']] = final_df_3.apply(
    lambda row: pd.Series(update_positions_and_consequence(row)),
    axis=1
)

condition_x = (final_df_3['Dataset'] == 'RHO_Wan_2019')
    
#setting ref/alt and transcript ref/alt to the same allele
final_df_3.loc[condition_x,'ref_allele'] = final_df_3.loc[condition_x,'transcript_ref']

final_df_3.loc[condition_x,'alt_allele'] = final_df_3.loc[condition_x,'transcript_alt']

/var/folders/1w/gtwrp7yj6c954_jnlv0s_jk40000gn/T/ipykernel_1420/3203218625.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_2_key_RHO['hgvs_c'] = pd_2_key['HGVSc'].str.extract(r':(.*)')


In [32]:
final_df_3.to_csv("~/Downloads/final_pillar_data_v6.csv", index = False)

In [80]:
import gzip

chromosome = []
position = []
ref_allele = []
alt_allele = []
Clnsig = []
clnvc = []
Gene = []
variant_type = []
temp_list = []
temp_list2 = []
final_list = []

def clinvar_38(file1):
    with gzip.open(file1, "rt") as my_file:
        for line in my_file:
            if line[0] != "#":
                reader = line.split("\t")
                chromosome.append(reader[0])
                position.append(reader[1])
                ref_allele.append(reader[3])
                alt_allele.append(reader[4])
                significance = str(reader[7]).find("CLNSIG=")
                var_type = str(reader[7]).find("CLNVC=")
                clnvcso = str(reader[7]).find("CLNVCSO")
                origin = str(reader[7]).find("ORIGIN")             
                gene = str(reader[7]).find("GENEINFO")
                mc = str(reader[7]).find("MC=")
                temp_list2.append(reader[7][int(mc): int(origin)])
                temp_list.append(reader[7][int(gene):int(mc)])
                Clnsig.append((reader[7][int(significance)+7:int(var_type)-1]))
                clnvc.append((reader[7][int(var_type)+6:int(clnvcso)-1]))

        for element in temp_list:
            colon = element.find(":")
            Gene.append(element[9:int(colon)])
                
        for element in temp_list2:
            dash = element.find("|")
            semi = element.find(";")
            variant_type.append(element[14:-1])

    
    for i in range(0,len(chromosome)):
            mini_list = []
            mini_list.append(chromosome[i])
            mini_list.append(position[i])
            mini_list.append(ref_allele[i])
            mini_list.append(alt_allele[i])
            mini_list.append(Clnsig[i])
            mini_list.append(clnvc[i])
            mini_list.append(Gene[i])
            mini_list.append(variant_type[i])
            final_list.append(mini_list)
   
    import csv
    
    import pandas as pd
    
    import os.path
    
    header = ["Chromosome","Genomic_coordinates","Ref_allele","Alt_allele","Significance","Variant","Gene","Variant_type"]

    save_path = "/Users/malvikatejura/Downloads"
    
    file_name2 = "clinvar_wg_10152024" + ".csv"
    
    saved = os.path.join(save_path, file_name2)


    with open(saved,"w",encoding = "UTF8", newline = '') as new_file:
        writer = csv.writer(new_file)
        writer.writerow(header)
        for element in final_list:
            writer.writerow(element)
            
            
    df1 = pd.read_csv(saved, sep = ",", header = 0, dtype = {"Chromosome": str , "Genomic_coordinates" : int})
    
    df1.to_csv(path_or_buf= file_name2)
            
clinvar_38("/Users/malvikatejura/Downloads/clinvar_10152024.vcf.gz")

KeyboardInterrupt: 

In [55]:
gh = pd.read_csv("~/Downloads/final_pillar_data_v6.csv")

hh = pd.read_csv("~/Downloads/clinvar_wg_10152024.csv")

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (4,11,13,14,15,18,21,22,23,34,37,38,39,40,41) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [119]:
hh['Genomic_coordinates'] = hh['Genomic_coordinates'].astype(float)
hh['Chromosome'] = hh['Chromosome'].astype(str)
gh['Chrom'] = gh['Chrom'].astype(str)

merged = pd.merge(gh, hh, left_on=[#'Gene', 
    'ref_allele','alt_allele','hg38_start','Chrom'], right_on = [#'Gene',
                                             "Ref_allele","Alt_allele",'Genomic_coordinates','Chromosome'], how='left')

columns_to_update_1 = ['clinvar_sig']
columns_with_fallback_1 = ['Significance']

for x_col, y_col in zip(columns_to_update_1, columns_with_fallback_1):
    merged[x_col] = merged[x_col].fillna(merged[y_col])

merged.drop(columns=['Chromosome', 'Genomic_coordinates', 'Ref_allele',
       'Alt_allele', 'Variant', 'Variant_type','Unnamed: 0',"Significance"], inplace=True)

merged.to_csv("~/Downloads/pillar_data_clinvar38_test_annotated.csv", index = False)

In [204]:
import gzip

chromosome = []
position = []
ref_allele = []
alt_allele = []
Clnsig = []
clnvc = []
Gene = []
variant_type = []
temp_list = []
temp_list2 = []
final_list = []

def clinvar_38(file1):
    with gzip.open(file1, "rt") as my_file:
        for line in my_file:
            if line[0] != "#":
                reader = line.split("\t")
                chromosome.append(reader[0])
                position.append(reader[1])
                ref_allele.append(reader[3])
                alt_allele.append(reader[4])
                significance = str(reader[7]).find("CLNSIG=")
                var_type = str(reader[7]).find("CLNVC=")
                clnvcso = str(reader[7]).find("CLNVCSO")
                origin = str(reader[7]).find("ORIGIN")             
                gene = str(reader[7]).find("GENEINFO")
                mc = str(reader[7]).find("MC=")
                temp_list2.append(reader[7][int(mc): int(origin)])
                temp_list.append(reader[7][int(gene):int(mc)])
                Clnsig.append((reader[7][int(significance)+7:int(var_type)-1]))
                clnvc.append((reader[7][int(var_type)+6:int(clnvcso)-1]))

        for element in temp_list:
            colon = element.find(":")
            Gene.append(element[9:int(colon)])
                
        for element in temp_list2:
            dash = element.find("|")
            semi = element.find(";")
            variant_type.append(element[14:-1])

    
    for i in range(0,len(chromosome)):
            mini_list = []
            mini_list.append(chromosome[i])
            mini_list.append(position[i])
            mini_list.append(ref_allele[i])
            mini_list.append(alt_allele[i])
            mini_list.append(Clnsig[i])
            mini_list.append(clnvc[i])
            mini_list.append(Gene[i])
            mini_list.append(variant_type[i])
            final_list.append(mini_list)
   
    import csv
    
    import pandas as pd
    
    import os.path
    
    header = ["Chromosome","Genomic_coordinates","Ref_allele","Alt_allele","Significance","Variant","Gene","Variant_type"]

    save_path = "/Users/malvikatejura/Downloads"
    
    file_name2 = "clinvar_wg_hg19_10152024" + ".csv"
    
    saved = os.path.join(save_path, file_name2)


    with open(saved,"w",encoding = "UTF8", newline = '') as new_file:
        writer = csv.writer(new_file)
        writer.writerow(header)
        for element in final_list:
            writer.writerow(element)
            
            
    df1 = pd.read_csv(saved, sep = ",", header = 0, dtype = {"Chromosome": str , "Genomic_coordinates" : int})
    
    df1.to_csv(path_or_buf= file_name2)
            
clinvar_38("/Users/malvikatejura/Downloads/clinvar_10152024_hg19.vcf.gz")

KeyboardInterrupt: 

In [134]:
fh = pd.read_csv("~/Downloads/pillar_data_clinvar38_test_annotated.csv")

lh = pd.read_csv("~/Downloads/clinvar_wg_hg19_10152024.csv")

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (4,11,13,14,15,18,21,22,23,34,37,38,39,40,41) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [135]:
lh['Chromosome'] = lh['Chromosome'].astype(str)
fh['Chrom'] = fh['Chrom'].astype(str)

merged_2 = pd.merge(fh, lh, left_on=[#'Gene', 
    'ref_allele','alt_allele','hg19_pos','Chrom'], right_on = [#'Gene',
                                           "Ref_allele","Alt_allele",'Genomic_coordinates','Chromosome'], how='left')

columns_to_update_2 = ['clinvar_sig']
columns_with_fallback_2 = ['Significance']

for x_col, y_col in zip(columns_to_update_1, columns_with_fallback_1):
    merged_2[x_col] = merged_2[x_col].fillna(merged_2[y_col])

merged_2.drop(columns=['Chromosome', 'Genomic_coordinates', 'Ref_allele',
       'Alt_allele', 'Variant', 'Variant_type','Unnamed: 0',"Significance"], inplace=True)

merged_2.to_csv("~/Downloads/pillar_data_clinvar38_19_annotated_final_v4.csv", index = False)

In [137]:
import pandas as pd

gnomad = pd.read_csv("~/Downloads/gnomad_v4_01242025.csv")

merged_2 = pd.read_csv("~/Downloads/pillar_data_clinvar38_19_annotated_final_v4.csv")

gnomad['Position'] = gnomad['Position'].astype('float64')
gnomad['Chromosome'] = gnomad['Chromosome'].astype('str')
gnomad['Reference'] = gnomad['Reference'].astype('str')
gnomad['Alternate'] = gnomad['Alternate'].astype('str')

merged_2['hg38_start'] = merged_2['hg38_start'].astype('float64')
merged_2['Chrom'] = merged_2['Chrom'].astype('str')
merged_2['ref_allele'] = merged_2['ref_allele'].astype('str')
merged_2['alt_allele'] = merged_2['alt_allele'].astype('str')


jk = pd.merge(
    merged_2,
    gnomad[['Chromosome', 'Position', 'Reference', 'Alternate', 'Allele Frequency']],
    left_on=['Chrom', 'hg38_start', 'ref_allele', 'alt_allele'],
    right_on=['Chromosome', 'Position', 'Reference', 'Alternate'],
    how='left'
)

jk['gnomad_MAF'] = jk['Allele Frequency']

# Drop extra columns from gnomad that came with the merge
columns_to_drop = ['Chromosome', 'Position', 'Reference', 'Alternate', 'Allele Frequency']
merged_2_cleaned = jk.drop(columns=columns_to_drop)

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (4,11,13,14,15,18,21,22,23,34,37,38,39,40,41,43) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [138]:
merged_2_cleaned.drop(columns=['Gene_y'], inplace=True)

merged_2_cleaned.drop(columns=['Gene'], inplace=True)

merged_2_cleaned.rename(columns={'Gene_x': 'Gene'}, inplace=True)

In [139]:
print(len(merged_2_cleaned['ID'].unique()))

269494


In [140]:
merged_2_cleaned['Flag'] = merged_2_cleaned.apply(
    lambda row: '*' if (
        row['consequence'] == 'synonymous_variant' and
        row['aa_ref'] != row['aa_alt'] and
        pd.notna(row['aa_alt']) and
        row['aa_alt'] != '='
    ) else '',
    axis=1
)

In [141]:
merged_2_cleaned.to_csv("~/Downloads/pillar_data_clinvar38_19_annotated_final_v8_expanded_012425.csv", index = False)

In [143]:
import pandas as pd 

# Load input data

d_data = pd.read_csv("~/Downloads/pillar_data_with_curation_v4.csv")

d_data_aa = d_data[d_data['nucleotide_or_aa'] == 'aa']

merged_3 = pd.read_csv("~/Downloads/pillar_data_clinvar38_19_annotated_final_v8_expanded_012425.csv")

merged_3_aa = merged_3[merged_3['nucleotide_or_aa'] == 'aa']

merged_3_nuc = merged_3[merged_3['nucleotide_or_aa'] == 'nucleotide']

# Define grouping columns and columns to condense
group_cols = ['ID']

columns_to_condense = [
    'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele',
    'transcript_pos', 'transcript_ref', 'transcript_alt',
    'clinvar_sig', 'hgvs_c','gnomad_MAF','consequence'
]

# Select only the columns to condense along with the grouping column
columns_to_aggregate = group_cols + columns_to_condense

# Perform the groupby and aggregation
condensed_df = merged_3_aa[columns_to_aggregate].groupby(group_cols).agg(
    lambda x: '^'.join(map(str, x))
).reset_index()

# Merge the condensed data back to the original DataFrame
d_data_condensed = d_data_aa.merge(
    condensed_df,
    on=group_cols,
    how='left',
    suffixes=('', '_condensed')
)

# List of columns to drop
columns_to_drop = [
    'hg38_start', 'ref_allele', 'alt_allele',
    'transcript_pos', 'transcript_ref', 'transcript_alt',
    'clinvar_sig', 'hgvs_c', 'gnomad_MAF', 'consequence', 'Dataset_tag'
]

# Drop the specified columns
d_data_condensed_2 = d_data_condensed.drop(columns=columns_to_drop)

# Rename columns to remove '_condensed' suffix
d_data_condensed_2.columns = [col.replace('_condensed', '') for col in d_data_condensed_2.columns]

d_data_condensed_3 = pd.concat([d_data_condensed_2, merged_3_nuc])

new_column_order = ['ID','Dataset', 'Gene', 'HGNC_id', 'Chrom', 'STRAND','hg19_pos', 'hg38_start','hg38_end',
       'ref_allele', 'alt_allele', 'auth_transcript_id', 'transcript_pos',
       'transcript_ref', 'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt',
       'hgvs_c', 'hgvs_p', 'consequence', 'auth_reported_score',
       'auth_reported_rep_score', 'auth_reported_func_class',
       'auth_reported_normal_min', 'auth_reported_normal_max',
       'auth_reported_abnormal_min', 'auth_reported_abnormal_max',
       'splice_measure', 'gnomad_MAF', 'clinvar_sig', 'clinvar_star',
       'clinvar_date_last_reviewed', 'nucleotide_or_aa', 'MaveDB URN',
       'Ensembl_transript_ID', 'Ref_seq_transcript_ID', 'Model_system',
       'Assay_type', 'Phenotype_measured', 'Phenotype_detail', 'IGVF_produced','Flag']

d_data_condensed_3 = d_data_condensed_3[new_column_order]

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (3,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,33,35,36,38,39,40) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (4,11,13,14,15,18,21,22,23,34,37,38,39,40,41,42) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [144]:
# Update the 'STRAND' column based on VEP annotation

pd_2_key_strand = pd_2_key.groupby('SYMBOL')['STRAND'].first().reset_index()

d_data_condensed_3['STRAND'] = d_data_condensed_3['Gene'].map(pd_2_key_strand.set_index('SYMBOL')['STRAND'])

In [145]:
d_data_condensed_3.to_csv("~/Downloads/pillar_data_condensed_01_24_25.csv", index = False)

In [146]:
print(len(d_data_condensed_3['ID'].unique()))

269494


In [147]:
#check if the dataset lengths match 

import pandas as pd

pp = pd.read_csv("~/Downloads/pillar_data_with_curation_v4.csv")

dff = pd.read_csv("~/Downloads/pillar_data_condensed_01_24_25.csv")

df1_lengths = pp.groupby('Dataset').size()
df2_lengths = dff.groupby('Dataset').size()

# Compare lengths
comparison = pd.DataFrame({'df1': df1_lengths, 'df2': df2_lengths}).fillna(0)
same_length = comparison[comparison['df1'] == comparison['df2']]
not_same_length = comparison[comparison['df1'] != comparison['df2']]
print("Groups with the same length:")
print(same_length)
print("Groups not the same length:")
print(not_same_length)

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (4,7,8,11,15,21,22,23,29,34,37,39,40,41) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


Groups with the same length:
                                          df1    df2
Dataset                                             
BAP1_Waters_2024                        18108  18108
BARD1_unpublished                        8681   8681
BRCA1_Adamovich_2022_Cisplatin           1427   1427
BRCA1_Adamovich_2022_HDR                 2271   2271
BRCA1_Findlay_2018                       3893   3893
BRCA2_Hu_2024                             462    462
BRCA2_Sahu_2023_exon13_Cisplatin          252    252
BRCA2_Sahu_2023_exon13_Olaparib           252    252
BRCA2_Sahu_2023_exon13_SGE                252    252
CALM1_CALM2_CALM3_Weile_2017             2980   2980
CTCF_unpublished                         4816   4816
DDX3X_Radford_2023_cLFC_day15           12776  12776
F9_Popp_2024_carboxy_F9_specific         9681   9681
F9_Popp_2024_carboxy_gla_motif           9681   9681
F9_Popp_2024_heavy_chain                 9681   9681
F9_Popp_2024_light_chain                 9681   9681
F9_Popp_2024_stre

In [148]:
test = pd.read_csv("~/Downloads/pillar_data_clinvar38_19_annotated_final_v8_expanded_012425.csv")

/Users/malvikatejura/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (4,11,13,14,15,18,21,22,23,34,37,38,39,40,41,42) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [151]:
test[test['Dataset'] == 'BAP1_Waters_2024'].to_csv("~/Downloads/BAP1_test.csv")

In [152]:
test.dtypes

ID                             object
Dataset                        object
Gene                           object
HGNC_id                         int64
Chrom                          object
STRAND                        float64
hg19_pos                      float64
hg38_start                    float64
hg38_end                      float64
ref_allele                     object
alt_allele                     object
auth_transcript_id             object
transcript_pos                 object
transcript_ref                 object
transcript_alt                 object
aa_pos                         object
aa_ref                         object
aa_alt                         object
hgvs_c                         object
hgvs_p                         object
consequence                    object
auth_reported_score            object
auth_reported_rep_score        object
auth_reported_func_class       object
auth_reported_normal_min      float64
auth_reported_normal_max      float64
auth_reporte